# Path3000 Perturbed Gene LCI Ranking

Rank perturbed genes by final likelihood contribution index (LCI), computed relative to the control condition from `results_path3000`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RESULTS_DIR = Path('/projects/b1042/GoyalLab/jaekj/KeepingScore/perturb-seq/model_keeping_score/results_path3000')
SUMMARY_CSV = RESULTS_DIR / 'path3000_ll_final_condition_summary.csv'
OUT_DIR = Path('/projects/b1042/GoyalLab/jaekj/KeepingScore/perturb-seq/Interpretability/knn_clustering_result')

OUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

summary = pd.read_csv(SUMMARY_CSV)
summary.head()

In [ ]:
control = summary.loc[summary['condition_name'].str.lower().eq('control')]
if control.empty:
    raise ValueError('No control condition found in condition_name.')
if len(control) > 1:
    raise ValueError('Multiple control rows found; inspect condition_name before computing relative LCI.')

control_mean = control['ll_final_mean'].iloc[0]
control_sem = control['ll_final_sem'].iloc[0]

ranked_lci = (
    summary.loc[~summary['condition_name'].str.lower().eq('control')]
    .copy()
    .assign(
        lci=lambda d: d['ll_final_mean'] - control_mean,
        lci_sem=lambda d: (d['ll_final_sem'] ** 2 + control_sem ** 2) ** 0.5,
        lci_ci95=lambda d: 1.96 * d['lci_sem'],
    )
    .sort_values('lci', ascending=False)
    .reset_index(drop=True)
)
ranked_lci['rank'] = ranked_lci.index + 1

ranked_lci.to_csv(OUT_DIR / 'path3000_perturbed_gene_lci_ranking.csv', index=False)
ranked_lci[['rank', 'condition_name', 'lci', 'lci_ci95', 'll_final_mean', 'll_final_sem']].head(20)

In [ ]:
fig_width = max(14, 0.18 * len(ranked_lci))
fig, ax = plt.subplots(figsize=(fig_width, 5.5), constrained_layout=True)

ax.errorbar(
    ranked_lci['condition_name'],
    ranked_lci['lci'],
    yerr=ranked_lci['lci_ci95'],
    fmt='o',
    markersize=4,
    linewidth=1,
    elinewidth=0.8,
    capsize=2,
    color='#2f6f8f',
    ecolor='#8aa9b8',
)
ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.7)
ax.set_xlabel('Perturbed gene, ranked by LCI')
ax.set_ylabel('LCI (mean final LL - control mean)')
ax.set_title('Path3000 Perturbed Gene LCI Ranking')
ax.tick_params(axis='x', labelrotation=90, labelsize=8)
sns.despine(ax=ax)

plot_path = OUT_DIR / 'path3000_perturbed_gene_lci_ranking.png'
fig.savefig(plot_path, dpi=300, bbox_inches='tight')
plot_path

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 8), constrained_layout=True)
top = ranked_lci.head(30).sort_values('lci')

ax.barh(top['condition_name'], top['lci'], xerr=top['lci_ci95'], color='#2f6f8f', ecolor='#8aa9b8')
ax.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.7)
ax.set_xlabel('LCI (mean final LL - control mean)')
ax.set_ylabel('Perturbed gene')
ax.set_title('Top 30 Path3000 Perturbed Genes by LCI')
sns.despine(ax=ax)

top_plot_path = OUT_DIR / 'path3000_top30_perturbed_gene_lci.png'
fig.savefig(top_plot_path, dpi=300, bbox_inches='tight')
top_plot_path